# SageMaker Training Job 실행

로컬 `run_pm.py` 테스트 통과 후, SageMaker Training Job으로 제출합니다.

In [ ]:
import yaml
import boto3
import sagemaker
from pathlib import Path
from urllib.parse import urlparse

CONF_DIR     = Path("./conf")
PROJECT_ROOT = Path(".")

def load_yaml(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

env_cfg  = load_yaml(CONF_DIR / "env.yml")
meta_cfg = load_yaml(CONF_DIR / "meta.yml")

In [ ]:
session    = boto3.Session()
sm_session = sagemaker.Session(boto_session=session)
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION     = env_cfg["region"]

# S3 conf 경로
CONF_S3_PATH = "s3://{bucket}/{env}/{user_id}/{project}/{experiment}/".format(
    bucket     = env_cfg["s3"]["conf_bucket"],
    env        = env_cfg["env"],
    user_id    = meta_cfg["user_id"],
    project    = meta_cfg["project"],
    experiment = meta_cfg["experiment"],
)

sm_cfg    = meta_cfg["sagemaker"]
IMAGE_URI = (
    f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
    f"/gs-automl-base-containers/{sm_cfg['env_name']}:{sm_cfg['image_tag']}"
)

SAGEMAKER_ROLE = sagemaker.get_execution_role()

print(f"CONF_S3_PATH : {CONF_S3_PATH}")
print(f"IMAGE_URI    : {IMAGE_URI}")

In [ ]:
# modeling.ipynb를 S3 conf 경로에 업로드 (run_pm.py가 S3에서 노트북을 가져옴)
s3_client = boto3.client("s3", region_name=REGION)

def sync_notebooks(meta_cfg: dict, conf_s3_path: str, project_root: Path) -> list:
    parsed = urlparse(conf_s3_path)
    bucket = parsed.netloc
    prefix = parsed.path.lstrip("/").rstrip("/") + "/"

    uploaded = []
    for nb in meta_cfg.get("notebooks", []):
        if not nb.get("sync_to_conf", False):
            continue
        local_path = (project_root / nb["local_path"]).resolve()
        if not local_path.exists():
            print(f"⚠️  Not found: {local_path}")
            continue
        key = prefix + local_path.name
        s3_client.upload_file(str(local_path), bucket, key)
        s3_uri = f"s3://{bucket}/{key}"
        uploaded.append({"name": nb["name"], "s3": s3_uri})
        print(f"✅ {nb['name']} → {s3_uri}")
    return uploaded

sync_notebooks(meta_cfg, CONF_S3_PATH, PROJECT_ROOT)

In [ ]:
from sagemaker.estimator import Estimator

estimator = Estimator(
    base_job_name     = sm_cfg["base_job_name"],
    image_uri         = IMAGE_URI,
    entry_point       = sm_cfg["entry_point"],
    role              = SAGEMAKER_ROLE,
    instance_type     = sm_cfg["instance_type"],
    instance_count    = sm_cfg["instance_count"],
    sagemaker_session = sm_session,
    hyperparameters   = {
        "conf-s3-path": CONF_S3_PATH,
        "work-dir":     sm_cfg["work_dir"],
    },
)

estimator.fit(wait=True, logs="All")